
# Unattended Luggage & Abandoned Object Detection

**AI-Based Crowd Surveillance System** — detects stationary bags/backpacks/suitcases
separated from their owners in transit hubs (stations, airports, waiting areas).

**Pipeline:** YOLO11 (detection) + ByteTrack (multi-object tracking) → owner–object
association → distance & stationary-duration analysis → countdown timer →
abandoned-object confirmation → alerts, evidence snapshots, CSV report, annotated
output video.

Run the cells **top to bottom**. Upload a CCTV/surveillance video when prompted in
Section 3, adjust thresholds in Section 4 if needed, then run the pipeline in
Section 8. All outputs (annotated video, snapshots, CSV report) are saved to
`/content/output/` and zipped for download at the end.


## 1. Install dependencies

In [ ]:

!pip install -q ultralytics==8.3.* opencv-python-headless==4.10.* lap==0.5.* pandas numpy scipy tqdm
print("Dependencies installed.")


## 2. Imports

In [ ]:

import os
import cv2
import math
import time
import json
import uuid
import zipfile
import numpy as np
import pandas as pd
from collections import deque
from datetime import datetime, timedelta
from dataclasses import dataclass, field

from ultralytics import YOLO
from tqdm.notebook import tqdm

from google.colab import files

print("Imports OK. Ultralytics version check:")
import ultralytics
print(ultralytics.__version__)



## 3. Upload input CCTV video

Run the cell below and select a video file (`.mp4`, `.avi`, `.mov`, etc.) from your
computer. If you'd rather use a video already sitting in `/content/`, just set
`VIDEO_PATH` manually in the next cell instead of uploading.


In [ ]:

os.makedirs("/content/input", exist_ok=True)

uploaded = files.upload()
VIDEO_PATH = None
for fname in uploaded.keys():
    src = f"/content/{fname}"
    dst = f"/content/input/{fname}"
    if os.path.exists(src):
        os.replace(src, dst)
    VIDEO_PATH = dst

assert VIDEO_PATH is not None, "No video uploaded. Set VIDEO_PATH manually below if needed."
print("Video ready at:", VIDEO_PATH)


## 4. Configuration — thresholds, paths, camera/location metadata

In [ ]:

CONFIG = {
    # --- Paths ---
    "video_path": VIDEO_PATH,
    "output_dir": "/content/output",
    "model_weights": "yolo11s.pt",   # YOLO11 small — good accuracy/speed tradeoff on Colab GPU

    # --- Site metadata (used in the CSV report) ---
    "camera_id": "CAM-01",
    "location": "Main Waiting Hall - Platform 3",
    # Real-world clock time corresponding to frame 0. Defaults to "now" — set this
    # to the video's actual recording start time for accurate CSV timestamps.
    "recording_start_datetime": datetime.now(),

    # --- Detection ---
    "confidence_threshold": 0.35,
    "iou_threshold": 0.5,
    "person_class_id": 0,
    "luggage_class_ids": [24, 26, 28],   # COCO: 24=backpack, 26=handbag, 28=suitcase
    "tracker_cfg": "bytetrack.yaml",     # ByteTrack via Ultralytics tracker integration

    # --- Owner association ---
    "association_distance_px": 160,   # max distance to link a bag to a nearby person on first sighting
    "attended_distance_px": 220,      # if ANY person stays within this distance, object is "attended"

    # --- Stationary detection ---
    "stationary_window_seconds": 3.0,     # how much recent history to look at
    "stationary_movement_px": 18,         # max centroid drift allowed within the window to call it "stationary"

    # --- Separation / abandonment timer ---
    "separation_time_threshold_seconds": 15,   # countdown length (use 300 = 5 min for real deployments)
    "min_track_age_seconds": 1.0,              # ignore very fresh/noisy tracks

    # --- Output video ---
    "draw_trails": False,
    "output_fps": None,   # None = use source video FPS
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs(f"{CONFIG['output_dir']}/snapshots", exist_ok=True)
print(json.dumps({k: str(v) for k, v in CONFIG.items()}, indent=2))


## 5. Load YOLO11 model

In [ ]:

model = YOLO(CONFIG["model_weights"])
print("Model loaded:", CONFIG["model_weights"])
print("Classes of interest -> person:", CONFIG["person_class_id"],
      "| luggage:", CONFIG["luggage_class_ids"])



## 6. Data structures for tracked people & objects

- `TrackedPerson` — lightweight record of a person track (centroid + last-seen time).
- `TrackedObject` — full state machine for each luggage/bag track: position history,
  stationary flag, associated owner, separation timer, status, and risk score.


In [ ]:

STATUS_ATTENDED   = "ATTENDED"
STATUS_MONITORING = "MONITORING"     # separated + stationary, timer running
STATUS_ABANDONED  = "ABANDONED"      # timer expired -> confirmed alert

CLASS_NAMES = {24: "backpack", 26: "handbag", 28: "suitcase"}

def euclidean(p1, p2):
    return math.hypot(p1[0] - p2[0], p1[1] - p2[1])

def bbox_centroid(box):
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2.0, (y1 + y2) / 2.0)

@dataclass
class TrackedPerson:
    track_id: int
    centroid: tuple
    bbox: tuple
    last_seen_frame: int
    last_seen_time: float

@dataclass
class TrackedObject:
    track_id: int
    class_id: int
    class_name: str
    first_seen_time: float
    bbox: tuple = None
    centroid: tuple = None
    confidence: float = 0.0
    history: deque = field(default_factory=lambda: deque())   # (time, centroid)
    owner_id: int = None
    is_stationary: bool = False
    separation_start: float = None
    status: str = STATUS_ATTENDED
    alerted: bool = False
    last_seen_time: float = 0.0
    last_seen_frame: int = 0
    last_owner_distance: float = None
    risk_score: float = 0.0


## 7. Core logic — stationary check, association, risk scoring, alerting

In [ ]:

def update_stationary_flag(obj: TrackedObject, now_t: float):
    obj.history.append((now_t, obj.centroid))
    window = CONFIG["stationary_window_seconds"]
    while obj.history and (now_t - obj.history[0][0]) > window:
        obj.history.popleft()

    if len(obj.history) < 2 or (now_t - obj.history[0][0]) < window * 0.6:
        # not enough history yet to make a confident call
        return

    pts = [c for _, c in obj.history]
    max_disp = 0.0
    ref = pts[0]
    for p in pts[1:]:
        d = euclidean(ref, p)
        if d > max_disp:
            max_disp = d
    obj.is_stationary = max_disp <= CONFIG["stationary_movement_px"]


def nearest_person(centroid, persons: dict):
    best_id, best_dist = None, float("inf")
    for pid, p in persons.items():
        d = euclidean(centroid, p.centroid)
        if d < best_dist:
            best_dist, best_id = d, pid
    return best_id, best_dist


def compute_risk_score(obj: TrackedObject, elapsed: float):
    time_ratio = min(elapsed / CONFIG["separation_time_threshold_seconds"], 1.0)
    dist_ratio = 0.0
    if obj.last_owner_distance is not None:
        dist_ratio = min(obj.last_owner_distance / CONFIG["attended_distance_px"], 1.0)
    score = (0.6 * time_ratio) + (0.2 * dist_ratio) + (0.2 * obj.confidence)
    return round(min(max(score, 0.0), 1.0), 3)


events = []          # rows for the final CSV report
snapshot_count = 0

def frame_timestamp(now_t):
    return CONFIG["recording_start_datetime"] + timedelta(seconds=now_t)


def log_event(obj: TrackedObject, frame, condition, now_t):
    # Append a CSV row and save an evidence snapshot for a detected condition.
    global snapshot_count
    snapshot_count += 1
    ts = frame_timestamp(now_t)
    snap_name = f"snapshot_{obj.track_id}_{condition.replace(' ', '_')}_{snapshot_count}.jpg"
    snap_path = os.path.join(CONFIG["output_dir"], "snapshots", snap_name)
    cv2.imwrite(snap_path, frame)

    events.append({
        "timestamp": ts.strftime("%Y-%m-%d %H:%M:%S"),
        "camera_id": CONFIG["camera_id"],
        "location": CONFIG["location"],
        "object_track_id": obj.track_id,
        "object_type": obj.class_name,
        "owner_track_id": obj.owner_id,
        "condition": condition,
        "confidence": round(obj.confidence, 3),
        "risk_score": obj.risk_score,
        "distance_to_owner_px": round(obj.last_owner_distance, 1) if obj.last_owner_distance is not None else None,
        "snapshot_file": snap_name,
    })



## 8. Main processing loop

For every frame:
1. Run YOLO11 + ByteTrack (`model.track(..., persist=True)`) filtered to person + luggage classes.
2. Update person tracks (position, last-seen).
3. Update each luggage track: history, stationary flag, nearest-person distance, owner
   association (first sighting), separation timer, status, risk score.
4. Draw bounding boxes, IDs, live countdown timer, and status color coding.
5. On timer expiry → confirm **ABANDONED**, log a CSV row + evidence snapshot + on-screen alert banner.
6. Write the annotated frame to the output video.


In [ ]:

cap = cv2.VideoCapture(CONFIG["video_path"])
assert cap.isOpened(), f"Could not open video: {CONFIG['video_path']}"

src_fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
out_fps = CONFIG["output_fps"] or src_fps
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

OUTPUT_VIDEO_PATH = os.path.join(CONFIG["output_dir"], "annotated_output.mp4")
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, out_fps, (width, height))

persons = {}          # track_id -> TrackedPerson (current frame)
objects_state = {}     # track_id -> TrackedObject (persists across frames)

target_classes = [CONFIG["person_class_id"]] + CONFIG["luggage_class_ids"]

COLOR_ATTENDED   = (0, 200, 0)
COLOR_MONITORING = (0, 200, 255)
COLOR_ABANDONED  = (0, 0, 255)
COLOR_PERSON     = (255, 180, 0)

frame_idx = 0
pbar = tqdm(total=total_frames if total_frames > 0 else None, desc="Processing")

results_stream = model.track(
    source=CONFIG["video_path"],
    conf=CONFIG["confidence_threshold"],
    iou=CONFIG["iou_threshold"],
    classes=target_classes,
    tracker=CONFIG["tracker_cfg"],
    persist=True,
    stream=True,
    verbose=False,
)

for result in results_stream:
    frame = result.orig_img.copy()
    now_t = frame_idx / src_fps

    persons = {}
    current_objects_frame = set()

    boxes = result.boxes
    if boxes is not None and boxes.id is not None:
        xyxy = boxes.xyxy.cpu().numpy()
        cls  = boxes.cls.cpu().numpy().astype(int)
        ids  = boxes.id.cpu().numpy().astype(int)
        confs = boxes.conf.cpu().numpy()

        # First pass: register all people for this frame
        for box, c, tid, conf in zip(xyxy, cls, ids, confs):
            if c == CONFIG["person_class_id"]:
                centroid = bbox_centroid(box)
                persons[tid] = TrackedPerson(
                    track_id=tid, centroid=centroid, bbox=tuple(box),
                    last_seen_frame=frame_idx, last_seen_time=now_t,
                )

        # Second pass: update luggage/object tracks
        for box, c, tid, conf in zip(xyxy, cls, ids, confs):
            if c not in CONFIG["luggage_class_ids"]:
                continue
            current_objects_frame.add(tid)
            centroid = bbox_centroid(box)

            if tid not in objects_state:
                owner_id, owner_dist = nearest_person(centroid, persons)
                if owner_dist > CONFIG["association_distance_px"]:
                    owner_id = None
                objects_state[tid] = TrackedObject(
                    track_id=tid, class_id=int(c),
                    class_name=CLASS_NAMES.get(int(c), str(c)),
                    first_seen_time=now_t, owner_id=owner_id,
                )

            obj = objects_state[tid]
            obj.bbox = tuple(box)
            obj.centroid = centroid
            obj.confidence = float(conf)
            obj.last_seen_time = now_t
            obj.last_seen_frame = frame_idx

            update_stationary_flag(obj, now_t)

            _, nearest_dist = nearest_person(centroid, persons) if persons else (None, float("inf"))
            obj.last_owner_distance = None if nearest_dist == float("inf") else nearest_dist

            track_age = now_t - obj.first_seen_time
            is_attended = (nearest_dist <= CONFIG["attended_distance_px"])

            if is_attended or track_age < CONFIG["min_track_age_seconds"]:
                obj.separation_start = None
                obj.status = STATUS_ATTENDED
                obj.alerted = False
                obj.risk_score = round(0.2 * obj.confidence, 3)
            else:
                if obj.is_stationary:
                    if obj.separation_start is None:
                        obj.separation_start = now_t
                    elapsed = now_t - obj.separation_start
                    obj.risk_score = compute_risk_score(obj, elapsed)
                    if elapsed >= CONFIG["separation_time_threshold_seconds"]:
                        obj.status = STATUS_ABANDONED
                        if not obj.alerted:
                            obj.alerted = True
                            log_event(obj, frame, "ABANDONED_OBJECT_CONFIRMED", now_t)
                    else:
                        obj.status = STATUS_MONITORING
                else:
                    # separated but object is still moving (likely being carried) -> not abandoned
                    obj.separation_start = None
                    obj.status = STATUS_MONITORING if obj.last_owner_distance is not None else STATUS_ATTENDED
                    obj.alerted = False

    # --- Draw annotations ---
    for pid, p in persons.items():
        x1, y1, x2, y2 = [int(v) for v in p.bbox]
        cv2.rectangle(frame, (x1, y1), (x2, y2), COLOR_PERSON, 1)
        cv2.putText(frame, f"P{pid}", (x1, max(0, y1 - 6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, COLOR_PERSON, 1)

    for tid in current_objects_frame:
        obj = objects_state[tid]
        x1, y1, x2, y2 = [int(v) for v in obj.bbox]
        color = {STATUS_ATTENDED: COLOR_ATTENDED,
                 STATUS_MONITORING: COLOR_MONITORING,
                 STATUS_ABANDONED: COLOR_ABANDONED}[obj.status]
        thickness = 3 if obj.status == STATUS_ABANDONED else 2
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)

        label = f"{obj.class_name} #{tid} [{obj.status}] conf:{obj.confidence:.2f}"
        cv2.putText(frame, label, (x1, max(0, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        if obj.status in (STATUS_MONITORING, STATUS_ABANDONED) and obj.separation_start is not None:
            elapsed = now_t - obj.separation_start
            remaining = max(0, CONFIG["separation_time_threshold_seconds"] - elapsed)
            mm, ss = divmod(int(remaining), 60)
            timer_txt = f"{mm:02d}:{ss:02d}" if obj.status == STATUS_MONITORING else "ALERT"
            cv2.putText(frame, timer_txt, (x1, y2 + 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
            cv2.putText(frame, f"risk:{obj.risk_score:.2f}", (x1, y2 + 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

        if obj.status == STATUS_ABANDONED:
            cv2.putText(frame, "!!! ABANDONED OBJECT ALERT !!!", (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, COLOR_ABANDONED, 2)

    cv2.putText(frame, f"{CONFIG['location']} | {frame_timestamp(now_t).strftime('%Y-%m-%d %H:%M:%S')}",
                (20, height - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    writer.write(frame)
    frame_idx += 1
    pbar.update(1)

pbar.close()
writer.release()
print("Processing complete.")
print("Annotated video saved to:", OUTPUT_VIDEO_PATH)
print("Total confirmed abandoned-object events:", len(events))


## 9. Generate CSV report

In [ ]:

CSV_PATH = os.path.join(CONFIG["output_dir"], "abandoned_object_report.csv")

if events:
    df = pd.DataFrame(events)
else:
    df = pd.DataFrame(columns=[
        "timestamp", "camera_id", "location", "object_track_id", "object_type",
        "owner_track_id", "condition", "confidence", "risk_score",
        "distance_to_owner_px", "snapshot_file",
    ])

df.to_csv(CSV_PATH, index=False)
print("CSV report saved to:", CSV_PATH)
df


## 10. Preview a sample annotated frame

In [ ]:

import matplotlib.pyplot as plt

cap = cv2.VideoCapture(OUTPUT_VIDEO_PATH)
mid_frame = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) // 2)
cap.set(cv2.CAP_PROP_POS_FRAMES, mid_frame)
ok, frame = cap.read()
cap.release()

if ok:
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("Sample annotated frame")
    plt.show()
else:
    print("Could not read a preview frame (video may be very short).")


## 11. Package outputs and download

In [ ]:

ZIP_PATH = "/content/unattended_luggage_outputs.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(OUTPUT_VIDEO_PATH, arcname="annotated_output.mp4")
    zf.write(CSV_PATH, arcname="abandoned_object_report.csv")
    snap_dir = os.path.join(CONFIG["output_dir"], "snapshots")
    for fname in os.listdir(snap_dir):
        zf.write(os.path.join(snap_dir, fname), arcname=f"snapshots/{fname}")

print("Packaged:", ZIP_PATH)
files.download(ZIP_PATH)



## 12. Notes & tuning guide

- **`separation_time_threshold_seconds`**: set to `15` here for quick demoing on short
  clips. For real deployments, standard practice is **300–600 seconds (5–10 min)**.
- **`association_distance_px` / `attended_distance_px`**: these are pixel distances, so
  they depend on camera resolution and how zoomed-in the scene is — recalibrate per camera.
- **`stationary_movement_px` / `stationary_window_seconds`**: controls sensitivity to
  camera jitter vs. genuine stillness. Increase the pixel tolerance for lower-res or
  noisy feeds.
- The **owner** recorded in the CSV is the person nearest to the bag at first detection;
  the live "attended" check instead uses the closest person in the current frame, which
  is what actually drives the alert logic and is more robust to ID switches from the tracker.
- Detection is restricted to COCO classes `backpack`, `handbag`, `suitcase`, and `person`.
  Swap in a fine-tuned YOLO11 model if you need broader luggage categories (trolley bags,
  boxes, etc.).
- Multiple simultaneous abandoned-object events are all logged as separate CSV rows with
  their own snapshot evidence.
